# Calibrate objective pair

Run after limits and orientation. Saved frames are already stage-aligned.


## Configure

Edit `session_id`, `job_name`, and `reference_slot`. The session identifier is
written into the measurement report and adopted calibration provenance. Use a
stable `calibration_name` per lens setup. You do not type objective names: the
driver resolves the configured reference slot's full hardware name and reports
it below; after you switch objectives it reads and reports the target slot/name.

Select `job_name` once in Navigator Expert and keep that same job selected for
the entire notebook. The notebook never changes the objective or the Navigator
Expert job: between reference and target steps, the operator switches **only the
objective**. Keep the XY scan-galvo and z-galvo settings unchanged between
objectives. Galvos are outside this calibration: the measured translation is
motoric X, motoric Y, and z-wide only. They do not need to be zero; they only need
to stay unchanged so they cancel from the reference-to-target delta.

Configure the XY images with a physical pixel size fine enough to support
sub-micrometre registration precision. The reference and target XY images must
have the **same physical pixel size** (µm/pixel), even though the objectives have
different magnifications; adjust the LAS X scan/zoom settings as needed. A coarse
pixel size cannot yield a reliable sub-micrometre translation, and the driver
refuses the comparison when the two physical pixel sizes differ.

Immediately before every frame or stack acquisition, the driver performs five
motoric-XY backlash jog-and-return passes and lands back at the same XY position
from the same direction. This does not touch Z, either galvo, the objective, or
the job, and it is operational only — backlash is not written into calibration
JSON.

The notebook otherwise changes as little microscope state as possible. The focus
steps only acquire the configured z-stacks and measure their incoming data. For
the later XY comparison only, the driver returns motoric XY to the recorded
reference location where required and sets z-wide to the focus just measured for
the active objective.

Calibration sits on top of orientation: the sideways offset between two
objectives is measured in the image and only becomes a stage offset because the
saved frames are already turned to the stage axes. If orientation has not been
measured yet (you have not run `set_orientation`), starting the session prints a
warning — run that notebook first, or the left/right result can come out turned.


In [ ]:
import _bootstrap
from navigator_expert.calibration.core import adopt
from navigator_expert.calibration.core import objective_pair as wf
from navigator_expert.config.machine import MACHINE

session = wf.start_session(
    session_id="2026-05-22_scope_calibration",
    job_name="Overview",
    reference_slot=1,
    sessions_root=MACHINE.snapshot_root() / "calibration",
    calibration_name="water_lens_setup",
)
session


## Reference focus

Keep the configured Navigator Expert job selected. Switch only to the reference
objective. Focus and configure its z-stack without changing the XY scan-galvo or
z-galvo settings, then run.


In [ ]:
session = wf.measure_parfocality_reference(session)


## Target focus

Keep the same Navigator Expert job selected and switch only to the target
objective. Do not adjust z-wide manually and do not change the XY scan-galvo or
z-galvo settings. Configure the target z-stack, then run.


In [ ]:
session = wf.measure_parfocality_target(session)


## Reference XY

Keep the same Navigator Expert job selected and switch only back to the reference
objective. Configure a fine physical pixel size (µm/pixel) that supports
sub-micrometre registration; the target XY acquisition must use that same
physical pixel size, although its scan/zoom setting may differ. Keep both galvo
settings unchanged. This cell returns motoric XY to the recorded reference
location and sets z-wide to the measured reference focus before acquiring; it
does not change the objective, job, or galvos.


In [ ]:
session = wf.measure_parcentricity_reference(session)


## Target XY

Keep the same Navigator Expert job selected and switch only to the target
objective. Do not move motoric XY manually. Match the reference XY acquisition's
physical pixel size in µm/pixel exactly; adjust the scan/zoom setting as needed
for this objective. Keep both galvo settings unchanged. This cell leaves motoric
XY at its post-switch position and sets z-wide to the measured target focus
before acquiring; it does not change the objective, job, or galvos.


In [ ]:
summary = wf.measure_parcentricity_target_and_save(session)
summary


## Adopt

Run only if the summary and overlay look right.


In [ ]:
adopt.adopt_calibration(
    session,
    session.objective_config_name,
    calibration_name=session.calibration_name,
    notebook_paths=[_bootstrap.NOTEBOOK_PATH],
)


## Check the adopted calibration (optional)

This validates the calibration you just adopted, at one spot. Park on a
textured area with the **reference** objective in place, focus, and run the
first cell; then switch **only the objective** to the target and run the
second. The check reads which objective is actually in at each step, so it
will refuse to run if you forgot to switch.

The second cell drives the stage (and z-wide) back to the same spot using
the adopted calibration's predicted offset — the same correction the driver
applies during an experiment when the active objective differs from the one
the origin was set under. The reported offset is therefore how far the
calibration is off at this spot: small is good; a large offset means re-run
the measurement steps above and adopt again. In the overlay, the right-hand
panel shows the two images after applying the measured leftover shift — a
good calibration already overlaps to white/grey in the left panel.


In [ ]:
from navigator_expert.calibration.core import calibration_check as check

check_session = check.start_session(
    session_id=session.session_id + "_check",
    job_name=session.job_name,
    sessions_root=MACHINE.snapshot_root() / "calibration",
    calibration_name=session.calibration_name,
)
check.measure_reference(check_session)


Switch only to the target objective (keep the same job selected), then run.

In [ ]:
check.measure_target_and_report(check_session)